## re Module Functions

In [1]:
# Beyond re.search(), the re module has several more functions. They fall into
# three categories:
# 1. Searching functions scan a search string for one or more regex matches.
# 2. Substitution functions replace the portions of a search string that match a regex.
# 3. Utility functions split a string on a regex or escape special characters.
import re

## Searching Functions

In [2]:
# re.search(<regex>, <string>, flags=0) scans a string for a regex match.
# It looks for the first location where the pattern matches, anywhere in the string.
re.search(r'(\d+)', 'foo123bar')
# <re.Match object; span=(3, 6), match='123'>
re.search(r'[a-z]+', '123FOO456', flags=re.IGNORECASE)
# <re.Match object; span=(3, 6), match='FOO'>
print(re.search(r'\d+', 'foo.bar'))
# None

# re.match(<regex>, <string>, flags=0) looks for a regex match at the beginning of a string.
# It's identical to re.search(), except that re.search() returns a match if <regex>
# matches anywhere in <string>, whereas re.match() returns a match only if <regex>
# matches at the beginning of <string>.
re.search(r'\d+', '123foobar')
# <re.Match object; span=(0, 3), match='123'>
re.search(r'\d+', 'foo123bar')
# <re.Match object; span=(3, 6), match='123'>
re.match(r'\d+', '123foobar')
# <re.Match object; span=(0, 3), match='123'>
print(re.match(r'\d+', 'foo123bar'))
# None

# re.fullmatch(<regex>, <string>, flags=0) looks for a regex match on an entire string.
# It returns a match only if <regex> matches <string> in its entirety.
print(re.fullmatch(r'\d+', '123foo'))
# None
print(re.fullmatch(r'\d+', 'foo123'))
# None
print(re.fullmatch(r'\d+', 'foo123bar'))
# None
re.fullmatch(r'\d+', '123')
# <re.Match object; span=(0, 3), match='123'>

# The same thing using re.search() with the ^ and $ anchors:
re.search(r'^\d+$', '123')
# <re.Match object; span=(0, 3), match='123'>

# re.findall(<regex>, <string>, flags=0) returns a list of all matches of a regex in a string.
# It returns strings, not match objects.
re.findall(r'\w+', '...foo,,,,bar:%$baz//|')
# ['foo', 'bar', 'baz']

# If <regex> contains a capturing group, then the list contains only the contents
# of the group, not the entire match.
re.findall(r'#(\w+)#', '#foo#.#bar#.#baz#')
# ['foo', 'bar', 'baz']

# If <regex> contains more than one capturing group, then re.findall() returns a
# list of tuples containing the captured groups.
re.findall(r'(\w+),(\w+)', 'foo,bar,baz,qux,quux,corge')
# [('foo', 'bar'), ('baz', 'qux'), ('quux', 'corge')]
re.findall(r'(\w+),(\w+),(\w+)', 'foo,bar,baz,qux,quux,corge')
# [('foo', 'bar', 'baz'), ('qux', 'quux', 'corge')]

# re.finditer(<regex>, <string>, flags=0) returns an iterator that yields regex matches.
# It scans the string for non-overlapping matches, like re.findall(), but it yields
# match objects instead of strings. That's useful in a for loop.
for i in re.finditer(r'\w+', '...foo,,,,bar:%$baz//|'):
    print(i)
# <re.Match object; span=(3, 6), match='foo'>
# <re.Match object; span=(10, 13), match='bar'>
# <re.Match object; span=(16, 19), match='baz'>

None
None
None
None
None
<re.Match object; span=(3, 6), match='foo'>
<re.Match object; span=(10, 13), match='bar'>
<re.Match object; span=(16, 19), match='baz'>


## Substitution Functions

In [3]:
# re.sub(<regex>, <repl>, <string>, count=0, flags=0) returns a new string that results
# from performing replacements on a search string.
# It finds the leftmost non-overlapping occurrences of <regex> in <string>, replaces
# each match as indicated by <repl>, and returns the result. <string> stays unchanged.

# Substitution by string: if <repl> is a string, then re.sub() inserts it into
# <string> in place of any sequences that match <regex>.
s = 'foo.123.bar.789.baz'
re.sub(r'\d+', '#', s)
# 'foo.#.bar.#.baz'
re.sub('[a-z]+', '(*)', s)
# '(*).123.(*).789.(*)'

# re.sub() substitutes numbered backreferences (\<n>) in <repl> with the text of
# the corresponding captured group.
re.sub(r'(\w+),bar,baz,(\w+)', r'\2,bar,baz,\1', 'foo,bar,baz,qux')
# 'qux,bar,baz,foo'

# You can also refer to named backreferences created with (?P<name><regex>) using
# the metacharacter sequence \g<name>.
re.sub(r'foo,(?P<w1>\w+),(?P<w2>\w+),qux', r'foo,\g<w2>,\g<w1>,qux', 'foo,bar,baz,qux')
# 'foo,baz,bar,qux'

# \g<n> also works for numbered groups. You need it when the replacement puts a
# digit right after the backreference: \10 would mean group 10, but \g<1>0 means
# group 1 followed by the character '0'.
re.sub(r'(\d+)', r'\g<1>0', 'foo 123 bar')
# 'foo 1230 bar'

# \g<0> refers to the text of the entire match.
re.sub(r'\d+', r'/\g<0>/', 'foo 123 bar')
# 'foo /123/ bar'

# If <regex> specifies a zero-length match, then re.sub() substitutes <repl> into
# every character position in the string.
re.sub('x*', '-', 'foo')
# '-f-o-o-'

# Substitution by function: if <repl> is a function, then re.sub() calls that
# function for each match found. It passes each match object as an argument to the
# function, and the function's return value is the replacement string.
def f(match_obj):
    s = match_obj.group(0)  # The matching string

    # s.isdigit() returns True if all characters in s are digits
    if s.isdigit():
        return str(int(s) * 10)
    else:
        return s.upper()

print(re.sub(r'\w+', f, 'foo.10.bar.20.baz.30'))
# FOO.100.BAR.200.BAZ.300

# Limiting the number of replacements: the count argument sets the maximum number
# of replacements to make.
re.sub(r'\w+', 'xxx', 'foo.bar.baz.qux')
# 'xxx.xxx.xxx.xxx'
re.sub(r'\w+', 'xxx', 'foo.bar.baz.qux', count=2)
# 'xxx.xxx.baz.qux'

# re.subn(<regex>, <repl>, <string>, count=0, flags=0) is identical to re.sub(),
# except that it returns a two-tuple consisting of the modified string and the
# number of substitutions made.
re.subn(r'\w+', 'xxx', 'foo.bar.baz.qux')
# ('xxx.xxx.xxx.xxx', 4)
re.subn(r'\w+', 'xxx', 'foo.bar.baz.qux', count=2)
# ('xxx.xxx.baz.qux', 2)
def f(match_obj):
    m = match_obj.group(0)
    if m.isdigit():
        return str(int(m) * 10)
    else:
        return m.upper()

print(re.subn(r'\w+', f, 'foo.10.bar.20.baz.30'))
# ('FOO.100.BAR.200.BAZ.300', 6)

FOO.100.BAR.200.BAZ.300
('FOO.100.BAR.200.BAZ.300', 6)


## Utility Functions

In [4]:
# re.split(<regex>, <string>, maxsplit=0, flags=0) splits a string into substrings,
# using <regex> as the delimiter, and returns the substrings as a list.
# Here the delimiter is a comma (,), semicolon (;), or slash (/), surrounded by any
# amount of whitespace.
re.split(r'\s*[,;/]\s*', 'foo,bar  ;  baz / qux')
# ['foo', 'bar', 'baz', 'qux']

# If <regex> contains capturing groups, then the return list includes the matching
# delimiter strings as well.
re.split(r'(\s*[,;/]\s*)', 'foo,bar  ;  baz / qux')
# ['foo', ',', 'bar', '  ;  ', 'baz', ' / ', 'qux']

# If you need groups in <regex> but don't want the delimiters in the list, use
# non-capturing groups (?:...).
re.split(r'(?:\s*[,;/]\s*)', 'foo,bar  ;  baz / qux')
# ['foo', 'bar', 'baz', 'qux']

# maxsplit sets the maximum number of splits. The remainder of the string ends up
# as the last element of the list.
re.split(r',\s*', 'foo, bar, baz, qux, quux, corge', maxsplit=3)
# ['foo', 'bar', 'baz', 'qux, quux, corge']
re.split(r',\s*', 'foo, bar, baz, qux, quux, corge')
# ['foo', 'bar', 'baz', 'qux', 'quux', 'corge']
re.split(r',\s*', 'foo, bar, baz, qux, quux, corge', maxsplit=0)
# ['foo', 'bar', 'baz', 'qux', 'quux', 'corge']

# A negative maxsplit means no splits at all.
re.split(r',\s*', 'foo, bar, baz, qux, quux, corge', maxsplit=-1)
# ['foo, bar, baz, qux, quux, corge']

# re.escape(<regex>) escapes characters in a regex.
# It returns a copy of <regex> with each nonword character (anything other than a
# letter, digit, or underscore) preceded by a backslash. This is useful when you
# want to match a string that contains special characters literally.
print(re.match('foo^bar(baz)|qux', 'foo^bar(baz)|qux'))
# None
re.escape('foo^bar(baz)|qux') == r'foo\^bar\(baz\)\|qux'
# True
re.match(re.escape('foo^bar(baz)|qux'), 'foo^bar(baz)|qux')
# <re.Match object; span=(0, 16), match='foo^bar(baz)|qux'>

None


<re.Match object; span=(0, 16), match='foo^bar(baz)|qux'>

## Compiled Regex Objects in Python

In [5]:
# re.compile(<regex>, flags=0) compiles a regex into a regular expression object.
# You can then use the object to do pattern matching over and over again.

# These two statements do the same thing:
#   re.search(<regex>, <string>, <flags>)
#   re.compile(<regex>, <flags>).search(<string>)

# And so do these:
#   re_obj = re.compile(<regex>, <flags>)
#   result = re_obj.search(<string>)
re.search(r'(\d+)', 'foo123bar')
# <re.Match object; span=(3, 6), match='123'>
re.compile(r'(\d+)').search('foo123bar')
# <re.Match object; span=(3, 6), match='123'>
re_obj = re.compile(r'(\d+)')
re.search(re_obj, 'foo123bar')
# <re.Match object; span=(3, 6), match='123'>
re_obj.search('foo123bar')
# <re.Match object; span=(3, 6), match='123'>

# Here's another example that uses the IGNORECASE flag.
r1 = re.search('ba[rz]', 'FOOBARBAZ', flags=re.I)
re_obj = re.compile('ba[rz]', flags=re.I)
r2 = re_obj.search('FOOBARBAZ')
r1
# <re.Match object; span=(3, 6), match='BAR'>
r2
# <re.Match object; span=(3, 6), match='BAR'>

<re.Match object; span=(3, 6), match='BAR'>

## Why Bother Compiling a Regex?

In [6]:
# If you use a regex over and over in your code, compiling it once and reusing the
# object saves repeating the same regex in lots of places. It also makes the code
# more modular: you can define the regex in one place and change it in one place.
# Python caches recently used regexes behind the scenes, so the performance
# difference is usually small. The main benefit is readability.
re_obj = re.compile(r'\d+')
re_obj.search('foo1bar')
# <re.Match object; span=(3, 4), match='1'>
re_obj.search('foo22bar')
# <re.Match object; span=(3, 5), match='22'>
print(re_obj.search('foobar'))
# None

None


## Regular Expression Object Methods

In [7]:
# A compiled regex object re_obj supports the same functions as the re module,
# but called as methods, without the <regex> argument:
#   re_obj.search(<string>[, <pos>[, <endpos>]])
#   re_obj.match(<string>[, <pos>[, <endpos>]])
#   re_obj.fullmatch(<string>[, <pos>[, <endpos>]])
#   re_obj.findall(<string>[, <pos>[, <endpos>]])
#   re_obj.finditer(<string>[, <pos>[, <endpos>]])
# These also accept optional <pos> and <endpos> arguments, which restrict the search
# to the slice of <string> from <pos> up to (but not including) <endpos>.
re_obj = re.compile(r'\d+')
s = 'foo123barbaz'
re_obj.search(s)
# <re.Match object; span=(3, 6), match='123'>
s[6:9]
# 'bar'
print(re_obj.search(s, 6, 9))
# None

# With <pos> set, ^ still means the true start of the string, not the start of
# the <pos> slice.
re_obj = re.compile('^bar')
s = 'foobarbaz'
s[3:]
# 'barbaz'
print(re_obj.search(s, 3))
# None

# But re_obj.match() anchors at <pos>, so it does find 'bar' starting at position 3.
re_obj.match(s, 3)
# None

# re_obj.findall() and re_obj.finditer() with <pos> and <endpos>:
re_obj = re.compile(r'\w+')
re_obj.findall('...foo,,,,bar:%$baz//|', 3, 12)
# ['foo', 'ba']

# re_obj.split(<string>, maxsplit=0)
# re_obj.sub(<repl>, <string>, count=0)
# re_obj.subn(<repl>, <string>, count=0)
# These behave like re.split(), re.sub(), and re.subn(), but don't take <pos> or <endpos>.
re_obj = re.compile(r',\s*')
re_obj.split('foo, bar, baz, qux')
# ['foo', 'bar', 'baz', 'qux']
re_obj = re.compile(r'\d+')
re_obj.sub('#', 'foo.123.bar.789.baz')
# 'foo.#.bar.#.baz'
re_obj.subn('#', 'foo.123.bar.789.baz')
# ('foo.#.bar.#.baz', 2)

None
None


('foo.#.bar.#.baz', 2)

## Regular Expression Object Attributes 

In [8]:
# re_obj.flags       Any <flags> that are in effect for the regex
# re_obj.groups      The number of capturing groups in the regex
# re_obj.groupindex  A dictionary mapping each symbolic group name to its group number
# re_obj.pattern     The <regex> pattern that produced this object
re_obj = re.compile(r'(?m)(\w+),(\w+)', re.I)
re_obj.flags
# 42
int(re.I | re.M | re.UNICODE)
# 42

# The value of re_obj.flags is the logical OR of the flags in effect: re.I (2),
# re.M (8), and re.UNICODE (32), which is on by default.
re_obj.groups
# 2
re_obj.pattern
# '(?m)(\\w+),(\\w+)'
re_obj = re.compile(r'(?P<w1>),(?P<w2>)')
re_obj.groupindex
# mappingproxy({'w1': 1, 'w2': 2})
re_obj.groupindex['w1']
# 1
re_obj.groupindex['w2']
# 2

2

## Match Object Methods and Attributes

In [9]:
# Most functions and methods in the re module return a match object when there's a
# successful match. Because a match object is truthy, you can use it in a conditional.
m = re.search('bar', 'foo.bar.baz')
m
# <re.Match object; span=(4, 7), match='bar'>
bool(m)
# True
if re.search('bar', 'foo.bar.baz'):
    print('Found a match')
# Found a match

Found a match


## Match Object Methods

In [10]:
# match.group([<group1>, ...]) returns the specified captured group(s) from a match.
# For numbered groups, match.group(n) returns the nth group.
m = re.search(r'(\w+),(\w+),(\w+)', 'foo,bar,baz')
m.group(1)
# 'foo'
m.group(3)
# 'baz'

# If you capture groups using (?P<name><regex>), then match.group(<name>) returns
# the corresponding named group.
m = re.match(r'(?P<w1>\w+),(?P<w2>\w+),(?P<w3>\w+)', 'quux,corge,grault')
m.group('w1')
# 'quux'
m.group('w3')
# 'grault'

# With multiple arguments, .group() returns a tuple of all the groups specified.
m.group('w1', 'w3')
# ('quux', 'grault')
m.group(1, 'w2', 3)
# ('quux', 'corge', 'grault')

# With no arguments, or with 0, .group() returns the entire match.
m.group()
# 'quux,corge,grault'
m.group(0)
# 'quux,corge,grault'

# If a group didn't participate in the match, .group() returns None.
print(re.search(r'(\w+),(\w+),(\w+)?', 'foo,bar,').group(3))
# None

# If a group matches multiple times, only the last match is accessible.
m = re.match(r'(\w{3},)+', 'foo,bar,baz,qux')
m.group(1)
# 'baz,'
m.group(0)
# 'foo,bar,baz,'

# match.__getitem__(<grp>) retrieves a captured group from a match.
# m.__getitem__(n) is the same as m.group(n), and it's what makes square-bracket
# indexing syntax work: m[n].
m = re.search(r'(\w+),(\w+),(\w+)', 'foo,bar,baz')
m.group(2)
# 'bar'
m.__getitem__(2)
# 'bar'
m[2]
# 'bar'
m = re.match(r'foo,(?P<w1>\w+),(?P<w2>\w+),qux', 'foo,bar,baz,qux')
m['w2']
# 'baz'

# match.groups(default=None) returns a tuple of all captured groups from a match.
m = re.search(r'(\w+),(\w+),(\w+)', 'foo,bar,baz')
m.groups()
# ('foo', 'bar', 'baz')

# Groups that don't participate in the match return None, unless you pass a
# default value to use instead.
m = re.search(r'(\w+),(\w+),(\w+)?', 'foo,bar,')
m.groups()
# ('foo', 'bar', None)
m.groups(default='---')
# ('foo', 'bar', '---')

# match.groupdict(default=None) returns a dictionary of named captured groups.
# The keys are the group names and the values are the matched text.
m = re.match(r'foo,(?P<w1>\w+),(?P<w2>\w+),qux', 'foo,bar,baz,qux')
m.groupdict()
# {'w1': 'bar', 'w2': 'baz'}
m.groupdict()['w2']
# 'baz'

# As with .groups(), you can specify a default for groups that don't participate.
m = re.match(r'foo,(?P<w1>\w+),(?P<w2>\w+)?,qux', 'foo,bar,,qux')
m.groupdict()
# {'w1': 'bar', 'w2': None}
m.groupdict(default='---')
# {'w1': 'bar', 'w2': '---'}

# match.expand(<template>) performs backreference substitutions from a match.
# It returns the string that results from substituting \<n> and \g<name> in
# <template>, the same way re.sub() does.
m = re.search(r'(\w+),(\w+),(\w+)', 'foo,bar,baz')
m.groups()
# ('foo', 'bar', 'baz')
m.expand(r'\2')
# 'bar'
m.expand(r'[\3] -> [\1]')
# '[baz] -> [foo]'
m = re.search(r'(?P<num>\d+)', 'foo123qux')
m.expand(r'--- \g<num> ---')
# '--- 123 ---'

# match.start([<grp>]) returns the starting index of the match or a specific group.
# match.end([<grp>]) returns the ending index (one past the last character).
s = 'foo123bar456baz'
m = re.search(r'\d+', s)
m.start()
# 3
m.end()
# 6
s[m.start():m.end()]
# '123'

# With a group argument, they return the indices for that group.
s = 'foo123bar456baz'
m = re.search(r'(\d+)\D*(?P<num>\d+)', s)
m.group(1)
# '123'
m.start(1), m.end(1)
# (3, 6)
s[m.start(1):m.end(1)]
# '123'
m.group('num')
# '456'
m.start('num'), m.end('num')
# (9, 12)

# If a group matches a null string, then .start() and .end() are equal. If a group
# doesn't participate in the match, they return -1.
m = re.search(r'foo(\d*)', 'foobar')
m.start(1), m.end(1)
# (3, 3)
m = re.search(r'(\w+),(\w+),(\w+)?', 'foo,bar,')
m.start(3), m.end(3)
# (-1, -1)

# match.span([<grp>]) returns the starting and ending indices of the match as a tuple.
# m.span(n) is the same as (m.start(n), m.end(n)).
m = re.search(r'(\d+)\D*(?P<num>\d+)', 'foo123bar456baz')
m.span()
# (3, 12)
m.span(1)
# (3, 6)
m.span('num')
# (9, 12)

None


(9, 12)

## Match Object Attributes

In [11]:
# match.pos and match.endpos contain the values of <pos> and <endpos> that were
# passed to the search. If they weren't passed, they're 0 and the length of the string.
re_obj = re.compile(r'\d+')
m = re_obj.search('foo123bar', 2, 7)
m
# <re.Match object; span=(3, 6), match='123'>
m.pos, m.endpos
# (2, 7)
m = re.search(r'\d+', 'foo123bar')
m.pos, m.endpos
# (0, 9)

# match.lastindex contains the index of the last captured group.
s = 'foo123bar456baz'
m = re.search(r'(\d+)\D*(\d+)', s)
m.lastindex
# 2
s[m.start(m.lastindex):m.end(m.lastindex)]
# '456'

# If the last group matched was nested inside another group, match.lastindex refers
# to the outer group that closed last.
re.search(r'(\w+)_(\w+)', 'foo_bar').lastindex
# 2
re.search(r'(\w+_(\w+))', 'foo_bar').lastindex
# 1
re.search(r'((\w+)_(\w+))', 'foo_bar').lastindex
# 1

# If the regex has no groups, or no group participated, match.lastindex is None.
print(re.search('bar', 'foobarbaz').lastindex)
# None
print(re.search(r'(foo)?', 'bar').lastindex)
# None

# match.lastgroup contains the name of the last captured group, if it has a name.
s = 'foo123bar456baz'
m = re.search(r'(?P<n1>\d+)\D*(?P<n2>\d+)', s)
m.lastgroup
# 'n2'

# It's None if the last group has no name.
m = re.search(r'(\d+)\D*(\d+)', s)
print(m.lastgroup)
# None

# match.re contains the regular expression object for the match: the same object
# you'd get from re.compile().
regex = r'(\w+),(\w+),(\w+)'
m1 = re.search(regex, 'foo,bar,baz')
m1.re
# re.compile('(\\w+),(\\w+),(\\w+)')
re_obj = re.compile(regex)
re_obj
# re.compile('(\\w+),(\\w+),(\\w+)')
re_obj is m1.re
# True

# You can call regex object methods on it directly.
m1.re.match('quux,corge,grault')
# <re.Match object; span=(0, 17), match='quux,corge,grault'>
m1.re.groups
# 3

# match.string contains the search string for the match.
m = re.search(r'(\w+),(\w+),(\w+)', 'foo,bar,baz')
m.string
# 'foo,bar,baz'
re_obj = re.compile(r'(\w+),(\w+),(\w+)')
m = re_obj.search('foo,bar,baz')
m.string
# 'foo,bar,baz'

# Even with <pos> and <endpos>, match.string holds the entire string, not the slice.
m = re_obj.search('..foo,bar,baz..', 2, 13)
m
# <re.Match object; span=(2, 13), match='foo,bar,baz'>
m.string
# '..foo,bar,baz..'

None
None
None


'..foo,bar,baz..'

## Conclusion

In [12]:
# You now know how to use:
# - all the functions in the re module: searching, substitution, and utility functions
# - re.compile() to make a precompiled regular expression object
# - match objects and their methods and attributes
# For more complex jobs, there's also the third-party regex module (pip install regex).
# It's backward compatible with re and adds extra features.